In [ ]:
# starting using GPT2 for training

# basic gpt2 training data

import json
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import torch
# 加载分词器和模型
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.to("cuda")
tokenizer.pad_token = tokenizer.eos_token
data_file = '/content/gdrive/MyDrive/DaqingWork/distilling-step-by-step/nf.jsonl'

def load_and_process_data(file_path):
    with open(file_path, 'r') as f:
        idx = 0
        input_texts = []
        output_texts = []
        for line in f:
            # if idx>100:
            #   break
            # idx += 1
            item = json.loads(line)
            query = item['query']
            docs = item['unsorted_docs']
            reason = item['reason']

            formatted_docs = [f"[{i+1}] {doc}" for i, doc[:100] in enumerate(docs)]
            input_text = f"I will provide you with {len(docs)} passages, each indicated by number identifier []. \nRank the passages based on their relevance to query: {query}. \
And give me the reason why you rank them that way. Here is the documents: {' '.join(formatted_docs)}"

            input_texts.append(input_text)
            output_texts.append(reason)

    # 编码文本
    input_encodings = tokenizer(input_texts, truncation=True, padding=True, max_length=1024)
    output_encodings = tokenizer(output_texts, truncation=True, padding=True, max_length=1024)

    return input_encodings, output_encodings

# 加载和处理数据
input_encodings, output_encodings = load_and_process_data(data_file)

# 自定义数据集类
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, input_encodings, output_encodings, device="cuda"):
        self.input_encodings = {key: torch.tensor(val).to(device) for key, val in input_encodings.items()}
        self.output_encodings = {key: torch.tensor(val).to(device) for key, val in output_encodings.items()}

    def __getitem__(self, idx):
      input_item = {key: torch.tensor(val[idx]) for key, val in self.input_encodings.items()}
      # 输出标签应该单独处理，因为它们是模型训练的目标
      input_item['labels'] = torch.tensor(self.output_encodings['input_ids'][idx])
      
      return input_item


    def __len__(self):
        return len(self.input_encodings['input_ids'])

# 创建数据集实例
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset = MyDataset(input_encodings, output_encodings, device=device)

# 训练参数配置
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    logging_dir='./logs',
    logging_steps=100,
    save_strategy="epoch",  # 或者 "steps" 根据你的需求
    save_total_limit=1,
)

# 数据整理器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

# 初始化Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset
)

# 开始微调
trainer.train()
# 04674a45b4a3e404cc9e77481f795437d773f857 wandb

# 保存模型
model.save_pretrained("./finetuned_model", safe_serialization=False)


In [1]:
# First identifiy the predictor

import warnings

import torch

class Predictor:
    def __init__(self, label_id_dict, pad_token_id, task_name, tokenizer, layer,
                 naive_class_embs=None,
                 naive_final_emb=None) -> None:
        self.naive_class_embs = naive_class_embs
        self.naive_final_emb = naive_final_emb
        self.label_id_dict = label_id_dict
        self.pad_token_id = pad_token_id
        self.task_name = task_name
        self.tokenizer = tokenizer
        self.layer = layer

        if task_name == 'sst2':
            self.prefix_idxs = [tokenizer.encode('Sentiment', add_special_tokens=False)[-1],
                                tokenizer.encode(':', add_special_tokens=False)[0]]
        elif task_name == 'agnews':
            self.prefix_idxs = [tokenizer.encode('Answer', add_special_tokens=False)[-1],
                                tokenizer.encode(':', add_special_tokens=False)[0]]
        elif task_name == 'trec':
            self.prefix_idxs = [tokenizer.encode(' Type', add_special_tokens=False)[-1],
                                tokenizer.encode(':', add_special_tokens=False)[0]]
        elif task_name == 'emo':
            self.prefix_idxs = [tokenizer.encode('Emotion', add_special_tokens=False)[-1],
                                tokenizer.encode(':', add_special_tokens=False)[0]]
        else:
            raise NotImplementedError(f"task_name: {task_name}")

    # def get_pos(self, inputs):
    #     label_id_dict = self.label_id_dict
    #     pad_token_id = self.pad_token_id
    #     final_pos = (inputs['input_ids'] != pad_token_id).int().sum(-1) - 1
    #     device = inputs['input_ids'].device
    #     bsz, sql = inputs['input_ids'].shape
    #     class_poss = []
    #     for idx in label_id_dict.values():
    #         class_idx = idx
    #         for offset, prefix_idx in enumerate(reversed(self.prefix_idxs)):
    #             class_idx += prefix_idx * 100000 ** (offset + 1)
    #         input_ids = inputs['input_ids'].detach().clone()
    #         input_ids[:, 1:] += inputs['input_ids'][:, :-1] * 100000
    #         input_ids[:, 2:] += inputs['input_ids'][:, :-2] * 100000 * 100000
    #         class_pos = torch.arange(sql, device=device).unsqueeze(0).repeat(bsz, 1)[
    #             input_ids == class_idx].squeeze()
    #         class_poss.append(class_pos)
    #     return class_poss, final_pos
    def get_pos(self, inputs):
        label_id_dict = self.label_id_dict
        pad_token_id = self.pad_token_id
        # 计算非填充部分的最终位置
        final_pos = (inputs['input_ids'] != pad_token_id).int().sum(-1) - 1
        device = inputs['input_ids'].device
        bsz, sql = inputs['input_ids'].shape
        
        # 针对每个类别和reason计算位置
        class_poss = []
        reason_poss = []  # 新增一个存储reason位置的列表
        for idx in label_id_dict.values():
            class_idx = idx
            # 此处代码计算类别位置
            # ...
            # 假设input_ids已经包含了经过tokenize的输入序列
            # 以下代码查找序列中每个类别标签的索引位置
            class_pos = (inputs['input_ids'] == class_idx).nonzero()
            # 现在class_pos包含了类别标签在输入序列中的所有位置
            class_poss.append(class_pos)

        # 假设reasons是一系列的特定token序列
        # 这里我们需要一个额外的token编码来标识reason的开始，这个token需要预先定义
        reason_start_token_id = self.tokenizer.encode('reason_start', add_special_tokens=False)[0]
        
        # 找到每个reason的开始位置
        for b in range(bsz):
            input_ids = inputs['input_ids'][b]
            # 简单搜索，假设'reason_start'不在句子中间出现
            reason_start_positions = (input_ids == reason_start_token_id).nonzero(as_tuple=True)[0]
            
            for pos in reason_start_positions:
                # 我们需要确定reason的结束位置，这可能需要更复杂的逻辑
                # 假设reason是连续的token，并且以'reason_end'结束
                reason_end = pos + 1
                while reason_end < sql and input_ids[reason_end] != self.tokenizer.eos_token_id:
                    reason_end += 1

                # 存储每个reason的位置范围
                reason_poss.append((pos, reason_end))
        
        return class_poss, final_pos, reason_poss

    def _cal_all_key_and_values_of_class(self, inputs, past_key_values, one_class_one_list=False,
                                         include_final=False):
        class_poss, final_pos = self.get_pos(inputs)

        if include_final:
            class_poss.append(final_pos)

        def get_vecs(ker_or_value, class_poss):
            batch_idx = torch.arange(inputs['input_ids'].shape[0])
            class_vecs = []
            for poss in class_poss:
                class_vec = ker_or_value[batch_idx, :, poss, :]
                class_vecs.append(class_vec.unsqueeze(-2))
            if not one_class_one_list:
                class_vecs = torch.cat(class_vecs, dim=-2)
            return class_vecs

        key_and_values = []
        for layer in range(0, self.layer):
            key_and_values.append(tuple([get_vecs(_, class_poss) for _ in past_key_values[layer]]))
        return key_and_values  # tuple of tuple of tensor (bsz, n_head, num_class, d_head)

    def cal_all_key_and_values_of_class(self, inputs, results, one_class_one_list=False,
                                        include_final=False):
        past_key_values = results.past_key_values
        key_and_values = self._cal_all_key_and_values_of_class(inputs, past_key_values,
                                                               one_class_one_list=one_class_one_list,
                                                               include_final=include_final)
        return key_and_values  # tuple of tuple of tensor (bsz, n_head, num_class, d_head)

    def get_attention(self, inputs, results, layer):
        class_poss, final_pos = self.get_pos(inputs)
        batch_idx = torch.arange(inputs['input_ids'].shape[0])
        scores = []
        for class_pos in class_poss:
            attention = results.attentions[layer][batch_idx, :, final_pos, class_pos]
            score = attention
            if class_pos.numel() == 1:
                score = score.sum(-1)
            else:
                score = score.sum()
            if inputs['input_ids'].shape[0] != 1:
                warnings.warn(f'Only support batch_size=1 now!')
            scores.append(score.unsqueeze(0))
        scores = torch.cat(scores, dim=0)
        return scores

    def cal_all_sim_attn(self, inputs, results):
        sims = []
        for layer in range(0, self.layer):
            sim = self.get_attention(inputs=inputs, results=results, layer=layer)
            sims.append(sim.unsqueeze(1))
        sims = torch.cat(sims, dim=1)
        sims = sims.reshape(inputs['input_ids'].shape[0], -1)
        return sims
Predictor()

TypeError: Predictor.__init__() missing 5 required positional arguments: 'label_id_dict', 'pad_token_id', 'task_name', 'tokenizer', and 'layer'

In [ ]:
# attention weight injection 
import warnings
from typing import Callable, Optional, List, Union
from functools import wraps, partial
import torch
from torch import nn
from torch.nn import functional as F
from transformers import PreTrainedModel
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

# 
class AttentionerManagerBase:
    def __init__(self, model: PreTrainedModel, predictor: Predictor, n_demo, device,n_head):
        self.n_demo = n_demo
        self.n_head = n_head
        self.device = device
        self.model = model
        self.attention_adapters = self.register_attentioner_to_model()
        self.model.forward = manager_decoractor(self)(self.model.forward)
        self.predictor = predictor

    @property
    def input_ids(self):
        return self._input_ids

    @input_ids.setter
    def input_ids(self, input_ids):
        self._input_ids = input_ids
        class_poss, final_poss = self.predictor.get_pos({'input_ids': input_ids})
        for attention_adapter in self.attention_adapters:
            attention_adapter.register_input_ids(input_ids)
            attention_adapter.class_poss = class_poss
            attention_adapter.final_poss = final_poss

    def register_input_ids(self, input_ids):
        self.input_ids = input_ids

    def register_attentioner_to_model(self):
        raise NotImplementedError

    def zero_grad(self, set_to_none=True):
        if set_to_none:
            for attention_adapter in self.attention_adapters:
                attention_adapter.params = None
        else:
            for attention_adapter in self.attention_adapters:
                attention_adapter.zero_grad(set_to_none=True)

    def grad_process(self, grad, use_abs=True):
        assert len(grad.shape) == 4
        grad = grad.sum(1)
        if use_abs:
            grad = abs(grad)
        return grad

    def grad(self, *args, **kwargs):
        grads = []
        for attention_adapter in self.attention_adapters:
            grads.append(self.grad_process(attention_adapter.params.grad, *args, **kwargs))
        return grads

    def params(self):
        params = []
        for attention_adapter in self.attention_adapters:
            params.append(attention_adapter.weight)
        return params


def manager_decoractor(manager: AttentionerManagerBase):
    def model_forward_decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            input_ids = kwargs.get('input_ids', None)
            if input_ids is None:
                input_ids = args[0]
            manager.register_input_ids(input_ids)
            return fn(*args, **kwargs)

        return wrapper

    return model_forward_decorator


class GPT2AttentionerManager(AttentionerManagerBase):
    def __init__(self, model: PreTrainedModel, n_demo, predictor: Predictor, device, n_head=1):
        super().__init__(model, predictor, n_demo, device,n_head=n_head)

    def register_attentioner_to_model(self):
        attention_adapters = []
        for i, layer in enumerate(self.model.transformer.h):
            attention_adapter = AttentionAdapter(n_demo=self.n_demo, device=self.device,
                                                 n_head=self.n_head)
            layer.attn._attn = partial(gpt2_attn, layer.attn,
                                       attention_adapter=attention_adapter)
            attention_adapters.append(attention_adapter)
        return attention_adapters



class AttentionAdapterBase(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.use_flag = True

    def forward(self, attn_weights):
        if self.use_flag:
            return self._forward(attn_weights)
        else:
            return attn_weights

    def _forward(self, attn_weights):
        raise NotImplementedError

    def register_input_ids(self, input_ids: torch.Tensor):
        self.input_ids = input_ids


class AttentionAdapter(AttentionAdapterBase):
    def __init__(self, n_demo, n_head, device, reasoning_info=None) -> None:
        super().__init__()
        self.n_demo = n_demo
        self.n_head = n_head
        self.weight = torch.nn.Parameter(torch.zeros((n_head, n_demo), requires_grad=True, device=device))
        # Additional attribute to hold reasoning information
        self.reasoning_info = reasoning_info
        self.class_poss = None
        self.final_poss = None

    def _forward(self, attn_weights):
        # Here, we need to incorporate the reasoning information into the attention reweighting
        class_poss = self.class_poss
        final_poss = self.final_poss
        
        # Assume reasoning_info is a tensor of shape [batch_size, seq_len, reasoning_dim]
        # which contains the encoded reasoning info for each position in the input sequence.
        reasoning_weights = self.compute_reasoning_weights(self.reasoning_info)
        
        weight = self.weight.exp()
        bsz, n_head, seq_len, _ = attn_weights.shape
        assert bsz == 1
        
        # Use reasoning_weights to adjust the original attention weights
        attn_weights = attn_weights * reasoning_weights
        
        mask_mat = torch.ones((1, n_head, seq_len, seq_len), device=attn_weights.device)
        mask_mat[:, :, final_poss, class_poss] = weight.reshape(1, self.n_head, self.n_demo)
        
        return attn_weights * mask_mat

    def compute_reasoning_weights(self, reasoning_info):
        # Define how to compute reasoning_weights from reasoning_info here
        # This could be a learned transformation or a fixed function
        reasoning_weights = self.reasoning_transform(reasoning_info)
        reasoning_weights = nn.functional.softmax(reasoning_weights, dim=-1)  # Softmax以确保权重的和为1
        return reasoning_weights


In [2]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Config

class ReasoningAttentionAdapter(nn.Module):
    def __init__(self, num_labels, reasoning_vector_size, config):
        super().__init__()
        self.num_labels = num_labels
        self.reasoning_vector_size = reasoning_vector_size
        self.reasoning_weights = nn.Parameter(torch.randn(num_labels, reasoning_vector_size))
        self.dense = nn.Linear(config.n_embd, reasoning_vector_size)
        self.activation = nn.Tanh()
    
    def forward(self, hidden_states, attention_mask):
        reasoning_weights = self.activation(self.dense(hidden_states))
        reasoning_weights = torch.matmul(reasoning_weights, self.reasoning_weights.t())
        attention_mask = attention_mask.unsqueeze(1).expand_as(reasoning_weights)
        reasoning_attention = reasoning_weights * attention_mask
        return reasoning_attention

# 创建模型配置
model_name = "gpt2"
config = GPT2Config.from_pretrained(model_name)

# 初始化推理权重适配器
reasoning_adapter = ReasoningAttentionAdapter(num_labels=5, reasoning_vector_size=768, config=config)

# 在GPT-2模型中引入推理权重适配器
class GPT2WithReasoning(GPT2LMHeadModel):
    def __init__(self, config):
        super().__init__(config)
        self.reasoning_adapter = reasoning_adapter
    
    def forward(self, input_ids, attention_mask, labels=None):
        transformer_outputs = self.transformer(input_ids, attention_mask=attention_mask)
        hidden_states = transformer_outputs.last_hidden_state
        
        # 应用推理适配器
        reasoning_attention = self.reasoning_adapter(hidden_states, attention_mask)
        
        # 这里可以根据需求对模型输出和推理注意力权重进行进一步的处理和组合
        
        # 原始模型输出
        lm_logits = self.lm_head(hidden_states)

        # 如果提供了标签，就计算损失
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(lm_logits.view(-1, self.config.vocab_size), labels.view(-1))

        output = (lm_logits,) + transformer_outputs[1:] + (reasoning_attention,)
        return ((loss,) + output) if loss is not None else output

# 接下来，你可以像往常一样初始化你的模型，并在微调中使用它。
model = GPT2WithReasoning(config=config)


/home/yuj49/anaconda3/envs/llama_factory/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pickle
import warnings
from dataclasses import dataclass, field
from typing import List
import os

from torch.optim import Adam
from tqdm import tqdm
from transformers.hf_argparser import HfArgumentParser
import torch
import torch.nn.functional as F
from ..lm_apis.lm_api_base import LMForwardAPI
from ..utils.data_wrapper import wrap_dataset, tokenize_dataset
from ..utils.load_huggingface_dataset import load_huggingface_dataset_train_and_test
from ..utils.random_utils import set_seed
from ..utils.other import load_args, set_gpu, sample_two_set_with_shot_per_class, dict_to
from transformers import Trainer, TrainingArguments, PreTrainedModel, AutoModelForCausalLM, \
    AutoTokenizer
from ..utils.load_local import convert_path_old, load_local_model_or_tokenizer, get_model_layer_num
from ..util_classes.arg_classes import ReweightingArgs
from ..utils.prepare_model_and_tokenizer import load_model_and_tokenizer, get_label_id_dict_for_args
from ..util_classes.predictor_classes import Predictor
from .attentioner_for_train import AttentionAdapter,  \
    GPT2AttentionerManager
from datasets import concatenate_datasets
from copy import deepcopy


def train(args: ReweightingArgs):
    if os.path.exists(args.save_file_name):
        return
    set_gpu(args.gpu)
    if args.sample_from == 'test':
        dataset = load_huggingface_dataset_train_and_test(args.task_name)
    else:
        raise NotImplementedError(f"sample_from: {args.sample_from}")

    model, tokenizer = load_model_and_tokenizer(args)
    args.label_id_dict = get_label_id_dict_for_args(args, tokenizer)

    model = LMForwardAPI(model=model, model_name=args.model_name, tokenizer=tokenizer,
                         device='cuda:0',
                         label_dict=args.label_dict)

    training_args = TrainingArguments("./output_dir", remove_unused_columns=False,
                                      per_gpu_eval_batch_size=args.batch_size,
                                      per_gpu_train_batch_size=args.batch_size)

    def prepare_analysis_dataset(seed):
        demonstration, train_samples = sample_two_set_with_shot_per_class(dataset['train'],
                                                                          args.demonstration_shot,
                                                                          args.train_num_per_class,
                                                                          seed,
                                                                          label_name='label',
                                                                          a_total_shot=args.demonstration_total_shot)
        if args.sample_from == 'test':
            if len(dataset['test']) < args.actual_sample_size:
                args.actual_sample_size = len(dataset['test'])
                warnings.warn(
                    f"sample_size: {args.sample_size} is larger than test set size: {len(dataset['test'])},"
                    f"actual_sample_size is {args.actual_sample_size}")
            test_sample = dataset['test'].shuffle(seed=seed).select(range(args.actual_sample_size))
            analysis_dataset = wrap_dataset(test_sample, demonstration, args.label_dict,
                                            args.task_name)
            analysis_dataset = tokenize_dataset(analysis_dataset, tokenizer)

            train_dataset = wrap_dataset(train_samples, demonstration, args.label_dict,
                                         args.task_name)
            train_dataset = tokenize_dataset(train_dataset, tokenizer)
        else:
            raise NotImplementedError(f"sample_from: {args.sample_from}")

        return analysis_dataset, train_dataset, demonstration

    ys = []
    for seed in args.seeds:
        analysis_dataset, train_dataset, demonstration = prepare_analysis_dataset(
            seed)

        training_args = TrainingArguments("./output_dir", remove_unused_columns=False,
                                          per_gpu_eval_batch_size=1,
                                          per_gpu_train_batch_size=1)
        trainer = Trainer(model=model, args=training_args)

        num_layer = get_model_layer_num(model=model.model, model_name=args.model_name)
        predictor = Predictor(label_id_dict=args.label_id_dict, pad_token_id=tokenizer.pad_token_id,
                              task_name=args.task_name, tokenizer=tokenizer, layer=num_layer)
        if args.model_name in ['gpt2-xl']:
            attentionermanger = GPT2AttentionerManager(model.model, len(demonstration),
                                                       predictor=predictor,
                                                       device=model.device, n_head = args.n_head)
        else:
            raise NotImplementedError(f"model_name: {args.model_name}")

        params = attentionermanger.params()
        optimizer = Adam(params, lr=args.lr)

        set_seed(seed)
        loss_list = []
        for epoch in tqdm(range(args.epoch_num)):
            loss_item = 0.
            train_dataset = train_dataset.shuffle()
            train_dataloader = trainer.get_eval_dataloader(train_dataset)
            for idx, data in enumerate(train_dataloader):
                data = dict_to(data, model.device)
                output = model(**data)
                label = data['labels']
                loss = F.cross_entropy(output['logits'], label)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                loss_item += loss.item()
            loss_list.append(loss_item / idx)
            average_loss = float(loss_item / idx)
            print(f'{average_loss}/{epoch}')

        y = trainer.predict(analysis_dataset, ignore_keys=['results'])

        for _ in attentionermanger.attention_adapters:
            _.use_flag = False
        y2 = trainer.predict(analysis_dataset, ignore_keys=['results'])

        ys.append((y,loss_list, params, y2, average_loss))

    os.makedirs(os.path.dirname(args.save_file_name), exist_ok=True)
    with open(args.save_file_name, 'wb') as f:
        pickle.dump([ys, ], f)

ImportError: attempted relative import with no known parent package